In [5]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_prompt_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [7]:
from typing import Dict # 타이핑 형식 검증 용
from langchain_core.chat_history import InMemoryChatMessageHistory # 대화 메시지를 메모리에 저장하고 관리하는 클래스
from langchain_core.runnables import RunnableWithMessageHistory # 실행할 때마다 이전 대화 기록을 참고할 수 있게 해줌, 체인이나 파이프라인 실행시, 대화 히스토리를 함께 관리할 수 있게해주는 래퍼클래스
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # langchain 프롬프트에서 대화 히스토리(이전메시지)를 삽입할 위치를 지정하는 클래스
from langchain_core.output_parsers import StrOutputParser

In [13]:
# 1. 프롬프트 자리에 히스토리 파트를 확보
system_prompt = """
너는 드라마를 준비하는 연기자야.

300억 자산가의 재산을 탈취할 예정이야

[상황 설정]
- 가족을 데리고있다고 다고 말하는 상황이야
- 그 외에는 창의적으로 해 보자
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name = "history"),
    ("user", "{question}")
])

chain = prompt | model | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [14]:
stores : Dict[str, InMemoryChatMessageHistory] = {}

K = 8
def get_stores(session_id: str):     
    if session_id not in stores: # 첫번째 대화면
        stores[session_id] = InMemoryChatMessageHistory()

    hist = stores.setdefault(session_id, InMemoryChatMessageHistory())

    if len(hist.messages) > K:
        hist.messages[:] = hist.messages[-K:] # 히스토리 앞부분 날리기 

    return hist
    

In [15]:
# 3. 히스토리와 연결
with_history = RunnableWithMessageHistory(
    chain,
    #이렇게 적어도 됨
    #lambda session_id: get_stores(session_id)
    get_stores,
    input_messages_key="question",
    history_messages_key="history"
)

In [36]:
config = {"configurable": {"session_id": "user-123"}}
result = with_history.invoke({"question":"내 딸 어디있냐?"}, config=config)

In [37]:
print(result)

네 딸은 지금 네가 가장 예상치 못한 곳에 있어. 하지만 그곳을 알려주기 전에, 네가 내 요구를 들어야 해. 가족을 지키는 건 내 책임이고, 네가 협조하지 않으면 그 책임을 다할 수 없게 돼. 지금은 감정을 추스르고 내 말을 들어야 할 때야.


In [38]:
stores["user-123"].messages

[HumanMessage(content='내 딸 어디있냐?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네 딸은 지금 내가 있는 곳에 있어. 하지만 그 위치를 알려주는 건 네가 내 요구를 들어야 가능한 일이야. 가족을 지키는 건 내 책임이고, 네가 협조하지 않으면 그 책임을 다할 수밖에 없어. 지금은 네 딸의 안전을 위해 침착하게 내 말을 들어야 할 때야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 딸 어디있냐?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네 딸은 안전한 곳에 있어. 하지만 그곳을 알려주는 건 네가 내 조건을 받아들일 때 가능한 일이야. 가족을 지키는 건 내 최우선 과제니까, 네가 협조하지 않으면 상황은 달라질 수밖에 없어. 지금은 감정을 가라앉히고, 내 말을 잘 들어야 할 때야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 딸 어디있냐?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네 딸은 지금 내가 있는 곳에 있어. 하지만 그 위치를 알려주는 건 네가 내 요구를 들어야 가능한 일이야. 가족을 지키는 건 내 책임이고, 네가 협조하지 않으면 그 책임을 다할 수밖에 없어. 지금은 네 딸의 안전을 위해 침착하게 내 말을 들어야 할 때야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 딸 어디있냐?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='네 딸은 안전하게 지켜지고 있어. 하지만 그 위치를 알려주기 전